In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import inchi as rd_inchi
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
import chembl_structure_pipeline as csp
from chembl_structure_pipeline import standardizer

RDLogger.DisableLog('rdApp.*')  # errors are captured and logged per stage instead

# ----------------------------- Configuration -----------------------------
# Exact mass cutoff (Da), applied before duplicate removal.
MAX_MOLECULAR_MASS = 1000

# Max standard deviation tolerated among replicates of the same structure
# (regression only). Interpreted in the unit set by USE_LOG10 below.
VARIANCE_THRESHOLD = 0.2

# True  -> log10, so VARIANCE_THRESHOLD = 0.2 means a ~1.58x spread.
# False -> natural log, so the same 0.2 means a ~1.22x spread, i.e. a much
# stricter criterion that sends more replicate groups to the discordant pile.
# Concordant groups are collapsed to the geometric mean, consistent with
# judging concordance in log space.
USE_LOG10 = True

# Enantiomers are merged when True.
REMOVE_STEREOCHEMISTRY = True

# Drops the row when nothing carbon-containing is left (e.g. [Na+].[Cl-]).
REQUIRE_CARBON = True

# False -> aromatic canonical SMILES (recommended: a stable identifier, since
# different aromaticity perceptions can yield different Kekule forms for the
# same molecule). Set to True only if a downstream descriptor requires it.
KEKULIZE_OUTPUT = False

_uncharger = rdMolStandardize.Uncharger()
_largest_fragment_chooser = rdMolStandardize.LargestFragmentChooser()
_tautomer_enumerator = rdMolStandardize.TautomerEnumerator()
_CARBON = Chem.MolFromSmarts('[#6]')


def _to_smiles(mol: Chem.Mol | None) -> str | None:
    """Canonical SMILES. Kekule output is optional and always preceded by Kekulize()."""
    if mol is None:
        return None
    if not KEKULIZE_OUTPUT:
        return Chem.MolToSmiles(mol)
    mol = Chem.Mol(mol)
    Chem.Kekulize(mol, clearAromaticFlags=True)
    return Chem.MolToSmiles(mol, kekuleSmiles=True)


def _canonical_or_self(smile: str) -> str:
    """Canonicalize for comparison, so plain recanonicalization isn't counted as a change."""
    mol = Chem.MolFromSmiles(smile) if isinstance(smile, str) else None
    return _to_smiles(mol) if mol is not None else str(smile)


# --------------------------------- Log -----------------------------------
class CurationLog:
    """Keeps REMOVED ROWS separate from MODIFIED STRUCTURES (row kept)."""

    def __init__(self) -> None:
        self.removed: dict[str, int] = {}
        self.modified: dict[str, int] = {}
        self.audit: list[pd.DataFrame] = []

    def add_removed(self, label: str, n: int) -> None:
        if n:
            self.removed[label] = self.removed.get(label, 0) + int(n)

    def add_modified(self, label: str, n: int) -> None:
        if n:
            self.modified[label] = self.modified.get(label, 0) + int(n)

    def add_audit(self, stage: str, before: pd.Series, after: pd.Series,
                  raw_input: bool = False) -> None:
        if raw_input:
            before = before.apply(_canonical_or_self)
        changed = before != after
        if not changed.any():
            return
        self.audit.append(pd.DataFrame({
            'stage': stage,
            'smiles_before': before[changed].values,
            'smiles_after': after[changed].values,
        }))
        self.add_modified(f'Structure changed in "{stage}"', int(changed.sum()))


def _apply_stage(df, log, stage, func, reason_col='removal_reason', raw_input=False):
    """Apply func(smiles) -> (new_smiles | None, reason | None).

    Returns (kept df, removed df). No row is counted as removed when the
    structure was merely modified.
    """
    before = df['final_smiles'].copy()
    results = before.apply(func)
    new_smiles = results.apply(lambda r: r[0])
    reasons = results.apply(lambda r: r[1])

    failed = new_smiles.isna()
    removed_df = df[failed].copy()
    if not removed_df.empty:
        removed_df[reason_col] = reasons[failed].values
        for reason, n in removed_df[reason_col].value_counts().items():
            log.add_removed(f'{stage}: {reason}', n)

    kept = df[~failed].copy()
    kept['final_smiles'] = new_smiles[~failed].values
    log.add_audit(stage, before[~failed], new_smiles[~failed], raw_input=raw_input)
    return kept.reset_index(drop=True), removed_df


# ------------------------------- Stages ----------------------------------
def prepare_structure(smile: str) -> tuple[str | None, str | None]:
    """Validate the SMILES and strip stereochemistry through the RDKit API."""
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None, 'invalid SMILES'
    if REMOVE_STEREOCHEMISTRY:
        Chem.RemoveStereochemistry(mol)
    return _to_smiles(mol), None


def standardize_and_get_parent(smile: str) -> tuple[str | None, str | None]:
    """standardize_molblock + get_parent_molblock (strips salt/solvate/isotope, neutralizes)."""
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None, 'invalid SMILES'
    try:
        std_block = standardizer.standardize_molblock(Chem.MolToMolBlock(mol))
        parent_block, exclude = standardizer.get_parent_molblock(std_block)
        if exclude:
            return None, 'flagged as exclude by ChEMBL'
        parent = Chem.MolFromMolBlock(parent_block)
        if parent is None or parent.GetNumAtoms() == 0:
            return None, 'empty or unparseable parent'
        return _to_smiles(parent), None
    except Exception as exc:
        return None, f'standardization error ({type(exc).__name__})'


def keep_largest_fragment(smile: str) -> tuple[str | None, str | None]:
    """Only acts on SMILES that still hold more than one fragment after get_parent."""
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None, 'invalid SMILES'
    if len(Chem.GetMolFrags(mol)) <= 1:
        return _to_smiles(mol), None
    chosen = _largest_fragment_chooser.choose(mol)
    if chosen is None:
        return None, 'could not pick the largest fragment'
    return _to_smiles(chosen), None


def drop_inorganic(smile: str) -> tuple[str | None, str | None]:
    """Drop the ROW when what is left is a pure salt/inorganic (e.g. [Na+].[Cl-])."""
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None, 'invalid SMILES'
    if mol.GetNumAtoms() == 0:
        return None, 'empty molecule'
    if REQUIRE_CARBON and not mol.HasSubstructMatch(_CARBON):
        return None, 'no carbon (pure salt/inorganic)'
    return _to_smiles(mol), None


def neutralize(smile: str) -> tuple[str | None, str | None]:
    """Safety net: neutralize charges that get_parent left behind."""
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None, 'invalid SMILES'
    try:
        return _to_smiles(_uncharger.uncharge(mol)), None
    except Exception as exc:
        return None, f'neutralization error ({type(exc).__name__})'


def canonicalize_tautomer(smile: str) -> tuple[str | None, str | None]:
    mol = Chem.MolFromSmiles(smile)
    if mol is None:
        return None, 'invalid SMILES'
    try:
        return _to_smiles(_tautomer_enumerator.Canonicalize(mol)), None
    except Exception as exc:
        return None, f'tautomer canonicalization error ({type(exc).__name__})'


def add_inchi_stage(df: pd.DataFrame, log: CurationLog):
    """MolToInchi returns an empty string on failure — isna() used to miss that."""
    def _inchi(smile):
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            return None
        value = rd_inchi.MolToInchi(mol)
        return value if value else None

    df['InChI'] = df['final_smiles'].apply(_inchi)
    df['InChIKey'] = df['InChI'].apply(
        lambda s: rd_inchi.InchiToInchiKey(s) if s else None)
    failed = df['InChI'].isna() | df['InChIKey'].isna()
    removed_df = df[failed].copy()
    log.add_removed('InChI: calculation failed', int(failed.sum()))
    return df[~failed].reset_index(drop=True), removed_df


def molecular_mass_stage(df, log, max_mass=MAX_MOLECULAR_MASS):
    def _mass(smile):
        mol = Chem.MolFromSmiles(smile)
        return rdMolDescriptors.CalcExactMolWt(mol) if mol is not None else None

    df['molecular_mass'] = df['final_smiles'].apply(_mass)
    failed = df['molecular_mass'].isna() | (df['molecular_mass'] > max_mass)
    removed_df = df[failed].copy()
    log.add_removed(f'Molecular mass > {max_mass}', int(failed.sum()))
    return df[~failed].reset_index(drop=True), removed_df


# ----------------------------- Duplicates --------------------------------
def _finish_dedup(df, keep, conc, disc, log):
    empty = pd.DataFrame(columns=df.columns)
    final = pd.concat(keep, ignore_index=True) if keep else empty.copy()
    rem_c = pd.concat(conc, ignore_index=True) if conc else empty.copy()
    rem_d = pd.concat(disc, ignore_index=True) if disc else empty.copy()
    log.add_removed('Concordant duplicates (1 row kept)', len(rem_c))
    log.add_removed('Discordant duplicates (whole group discarded)', len(rem_d))
    return final, rem_c, rem_d


def remove_duplicates_classification(df, outcome_col, log):
    keep, conc, disc = [], [], []
    for _, group in df.groupby('InChIKey', sort=False):
        if len(group) == 1:
            keep.append(group)
        elif group[outcome_col].nunique() == 1:
            keep.append(group.iloc[[0]])
            conc.append(group.iloc[1:])
        else:
            disc.append(group)
    return _finish_dedup(df, keep, conc, disc, log)


def remove_duplicates_regression(df, target_col, log, threshold=VARIANCE_THRESHOLD):
    """Concordance judged in log space; representative = geometric mean."""
    df[target_col] = pd.to_numeric(df[target_col], errors='coerce')

    bad = df[target_col].isna()
    log.add_removed(f'Non-numeric value in "{target_col}"', int(bad.sum()))
    removed_invalid = df[bad].copy()
    df = df[~bad].reset_index(drop=True)

    nonpos = df[target_col] <= 0
    log.add_removed(f'Value <= 0 in "{target_col}"', int(nonpos.sum()))
    removed_invalid = pd.concat([removed_invalid, df[nonpos]], ignore_index=True)
    df = df[~nonpos].reset_index(drop=True)

    log_fn = np.log10 if USE_LOG10 else np.log
    df['log_target'] = log_fn(df[target_col])
    df['n_replicates'] = 1

    keep, conc, disc = [], [], []
    for _, group in df.groupby('InChIKey', sort=False):
        if len(group) == 1:
            keep.append(group)
            continue
        if group['log_target'].std(ddof=0) <= threshold:
            rep = group.iloc[[0]].copy()
            mean_log = group['log_target'].mean()
            rep[target_col] = (10 ** mean_log) if USE_LOG10 else np.exp(mean_log)
            rep['log_target'] = mean_log
            rep['n_replicates'] = len(group)
            keep.append(rep)
            conc.append(group.iloc[1:])
        else:
            disc.append(group)

    out, rem_c, rem_d = _finish_dedup(df, keep, conc, disc, log)
    return out, rem_c, rem_d, removed_invalid


# -------------------------------- Output ---------------------------------
def _save(df, path, label):
    if df is not None and not df.empty:
        df.to_csv(path, index=False)
        print(f"  -> {label}: {len(df)} rows in '{path}'")


def write_log(savepath: Path, log: CurationLog, initial: int, final: int) -> None:
    total_removed = sum(log.removed.values())
    lines = ['Curation log', '=' * 60,
             f'RDKit {rdkit.__version__} | chembl_structure_pipeline {csp.__version__}',
             f'Initial compounds: {initial}',
             '', 'REMOVED ROWS', '-' * 60]
    lines += [f'{k}: {v}' for k, v in log.removed.items()] or ['(none)']
    lines += ['-' * 60,
              f'Total rows removed: {total_removed}',
              '',
              f'Final compounds: {final}',
              f'Sanity check (initial - removed == final): '
              f'{initial - total_removed == final}',
              '',
              'MODIFIED STRUCTURES (row kept, not counted as a removal)',
              '-' * 60]
    lines += [f'{k}: {v}' for k, v in log.modified.items()] or ['(none)']
    (savepath / 'curation_log.txt').write_text('\n'.join(lines) + '\n')
    print('\n'.join(lines))


def curate_dataset(df, smiles_col, outcome_col, task_type='classification',
                   savepath='curated_data'):
    out_dir = Path(savepath)
    out_dir.mkdir(parents=True, exist_ok=True)
    log = CurationLog()
    initial = len(df)
    removed_parts = []

    df = df.copy()
    missing = df[smiles_col].isna() | df[outcome_col].isna()
    log.add_removed('Missing SMILES or outcome', int(missing.sum()))
    removed_parts.append(df[missing].assign(removal_reason='missing SMILES/outcome'))
    df = df[~missing].reset_index(drop=True)
    df['final_smiles'] = df[smiles_col]

    stages = [
        ('Preparation', prepare_structure),
        ('ChEMBL standardization + parent', standardize_and_get_parent),
        ('Largest fragment', keep_largest_fragment),
        ('Inorganic filter', drop_inorganic),
        ('Neutralization', neutralize),
        ('Canonical tautomer', canonicalize_tautomer),
    ]
    for i, (name, func) in enumerate(stages):
        df, removed = _apply_stage(df, log, name, func, raw_input=(i == 0))
        removed_parts.append(removed)
        print(f'{name}: {len(df)} compounds remaining')

    df, removed = add_inchi_stage(df, log)
    removed_parts.append(removed)
    df, removed = molecular_mass_stage(df, log)
    removed_parts.append(removed)

    if task_type == 'classification':
        df, rem_c, rem_d = remove_duplicates_classification(df, outcome_col, log)
    elif task_type == 'regression':
        df, rem_c, rem_d, rem_inv = remove_duplicates_regression(df, outcome_col, log)
        removed_parts.append(rem_inv)
    else:
        raise ValueError("task_type must be 'classification' or 'regression'")

    final = len(df)
    _save(rem_c, out_dir / 'removed_concordant_duplicates.csv', 'concordant duplicates')
    _save(rem_d, out_dir / 'removed_discordant_duplicates.csv', 'discordant duplicates')
    nonempty = [p for p in removed_parts if not p.empty]
    _save(pd.concat(nonempty, ignore_index=True) if nonempty else pd.DataFrame(),
          out_dir / 'removed_rows.csv', 'removed rows')
    _save(pd.concat(log.audit, ignore_index=True) if log.audit else pd.DataFrame(),
          out_dir / 'modified_structures.csv', 'modified structures')

    df = df.drop(columns=[smiles_col], errors='ignore')
    df.to_csv(out_dir / 'curated_dataset.csv', index=False)
    write_log(out_dir, log, initial, final)
    return df, log